# X-Ray Analiz Projesi - AI Servisi
Bu notebook, X-Ray görüntülerini analiz eden derin öğrenme modellerini çalıştırır ve dış dünyaya bir API sunar.

In [ ]:
# 1. Gerekli Kütüphanelerin Kurulumu
!pip install pyngrok fastapi uvicorn python-multipart

In [ ]:
# 2. Google Drive Bağlantısı
from google.colab import drive
import os

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
    print("Google Drive bağlandı.")

In [ ]:
# 3. Ayarlar ve Model Yolları
from pyngrok import conf, ngrok

# Ngrok Token
NGROK_TOKEN = "37nfbRSTCbHSVZ8JS4Q4XPemS6v_86DXzhAAn7XuxPHJZ3VuP"
conf.get_default().auth_token = NGROK_TOKEN

# Drive Klasör Yolu (Lütfen modelleri bu klasöre yükleyin)
DRIVE_BASE_PATH = "/content/drive/MyDrive/XRay_Models"

DEFAULT_MODEL_KEY = 'swin_t'

MODEL_CONFIGS = {
    'swin_t': {'file_path': f'{DRIVE_BASE_PATH}/best_swin_t_fold_2.pth', 'arch': 'swin_t', 'num_classes': 5},
    'vit_b_16': {'file_path': f'{DRIVE_BASE_PATH}/best_vit_b_16_fold_3.pth', 'arch': 'vit_b_16', 'num_classes': 5},
    'resnet50': {'file_path': f'{DRIVE_BASE_PATH}/best_resnet50_fold_4.pth', 'arch': 'resnet50', 'num_classes': 5},
    'vgg16': {'file_path': f'{DRIVE_BASE_PATH}/best_vgg16_fold_2.pth', 'arch': 'vgg16', 'num_classes': 5},
    'chexnet': {'file_path': f'{DRIVE_BASE_PATH}/best_chexnet_fold_1.pth', 'arch': 'densenet121', 'num_classes': 5},
    'inception_v3': {'file_path': f'{DRIVE_BASE_PATH}/best_inception_v3_fold_1.pth', 'arch': 'inception_v3', 'num_classes': 5}
}

In [ ]:
# 4. Model Sınıfları ve Yardımcı Fonksiyonlar
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import io

# Cihaz seçimi
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

class DenseNet121(nn.Module):
    def __init__(self, num_classes):
        super(DenseNet121, self).__init__()
        self.model = models.densenet121(weights=None)
        self.model.classifier = nn.Linear(self.model.classifier.in_features, num_classes)

    def forward(self, x):
        return self.model(x)

def get_model(arch_type, num_classes):
    if arch_type == 'swin_t':
        model = models.swin_t(weights=None)
        model.head = nn.Linear(model.head.in_features, num_classes)
    elif arch_type == 'vit_b_16':
        model = models.vit_b_16(weights=None)
        model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
    elif arch_type == 'resnet50':
        model = models.resnet50(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif arch_type == 'vgg16':
        model = models.vgg16(weights=None)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    elif arch_type == 'densenet121':
        model = DenseNet121(num_classes)
    elif arch_type == 'inception_v3':
        model = models.inception_v3(weights=None)
        model.aux_logits = False
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    else:
        raise ValueError(f"Bilinmeyen mimari: {arch_type}")
    return model.to(device)

current_model = None
current_model_key = None

def load_model(model_key: str):
    global current_model, current_model_key
    
    if current_model_key == model_key and current_model is not None:
        return current_model
    
    if model_key not in MODEL_CONFIGS:
        raise ValueError(f"Model bulunamadı: {model_key}")
    
    config = MODEL_CONFIGS[model_key]
    print(f">>> Model yükleniyor: {model_key}...")
    
    # Eski modeli temizle
    if current_model is not None:
        del current_model
        torch.cuda.empty_cache()
    
    # Yeni modeli yükle
    model = get_model(config['arch'], config['num_classes'])
    state_dict = torch.load(config['file_path'], map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    
    current_model = model
    current_model_key = model_key
    print(f">>> {model_key} başarıyla yüklendi!")
    return model

# İlk modeli yükle
try:
    load_model(DEFAULT_MODEL_KEY)
except Exception as e:
    print(f"Başlangıç modeli yüklenemedi: {e}")
    print("Lütfen model dosyalarının Drive'da doğru yerde olduğundan emin olun.")

# Görüntü Dönüşümleri
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_inception = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

class_names = ['Covid', 'Lung Opacity', 'Normal', 'Pneumonia', 'Viral Pneumonia']

In [ ]:
# 5. API Servisi Başlatma
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
import nest_asyncio
import threading
from typing import Optional

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/")
def read_root():
    return {"status": "ok", "active_model": current_model_key}

@app.get("/models")
def get_models():
    return {
        "active_model": current_model_key,
        "available_models": list(MODEL_CONFIGS.keys())
    }

@app.post("/predict")
async def predict_image(file: UploadFile = File(...), model_name: Optional[str] = Form(None)):
    global current_model
    
    requested_model = model_name if model_name else current_model_key
    
    # Model değişimi gerekiyorsa
    if requested_model != current_model_key:
        try:
            load_model(requested_model)
        except Exception as e:
            return {"error": f"Model yüklenemedi: {str(e)}"}
            
    try:
        image_data = await file.read()
        image = Image.open(io.BytesIO(image_data)).convert("RGB")
        
        # Inception özel transform
        if current_model_key == 'inception_v3':
            input_tensor = transform_inception(image).unsqueeze(0).to(device)
        else:
            input_tensor = transform(image).unsqueeze(0).to(device)
        
        with torch.no_grad():
            output = current_model(input_tensor)
            probs = torch.nn.functional.softmax(output, dim=1)
            conf, predicted = torch.max(probs, 1)
            
        predicted_class = class_names[predicted.item()]
        confidence = conf.item()
        
        return {
            "className": predicted_class,
            "confidence": confidence,
            "source_model": MODEL_CONFIGS[current_model_key]['arch']
        }
    except Exception as e:
        return {"error": str(e)}

# Ngrok Tüneli ve Server Başlatma
public_url = ngrok.connect(8000).public_url
print(f"\n\n🚀 LİNKİNİZ: {public_url}/predict \n\n")

nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000)